In [0]:
# 00 — Sanity Check
# Validates the cohort and label tables
# Run after build_silver.py

In [0]:
CATALOG = "general_scratch_catalog"
SCHEMA  = "general_scratch"
QUAL    = f"{CATALOG}.{SCHEMA}.ed_silver_subscription_terms_qualified"
TERMS  = f"{CATALOG}.{SCHEMA}.ed_silver_subscription_all_terms"
SUBS    = f"{CATALOG}.{SCHEMA}.ed_silver_subscriptions"
LABELS  = f"{CATALOG}.{SCHEMA}.ed_silver_subscription_term_start_labels"
EVENTS = f"{CATALOG}.{SCHEMA}.ed_silver_subs_kafka__events"
PLAN_TERMS = f"{CATALOG}.{SCHEMA}.ed_silver_subscription_plan_terms"


spark.conf.set("eda.qual",    QUAL)
spark.conf.set("eda.terms", TERMS)
spark.conf.set("eda.subs",    SUBS)
spark.conf.set("eda.labels",  LABELS)
spark.conf.set("eda.events",  EVENTS)
spark.conf.set("eda.plan_terms",  PLAN_TERMS)

## 1. Cohort size

In [0]:
%sql
SELECT
    COUNT(*)                        AS total_terms,
    COUNT(DISTINCT subscription_id) AS unique_subscriptions
FROM ${eda.qual}

In [0]:
%sql
SELECT
    subscription_id,
    COUNT(DISTINCT subscription_term_id) AS unique_subscriptions
FROM ${eda.qual}
GROUP BY subscription_id HAVING COUNT(DISTINCT subscription_term_id) > 1

In [0]:
%sql
select
*
from ${eda.terms}
where subscription_id = '02548d20-74b4-4256-9244-01466dc3c318'

In [0]:
%sql
select
raw_occurred_at,
occurred_at,
event_name,
old_renewal_at,
new_renewal_at
from ${eda.events}
where subscription_id = '02548d20-74b4-4256-9244-01466dc3c318'
order by 1

subscription `02548d20-74b4-4256-9244-01466dc3c318` has two terms even though he didn't churn and reactivate. exclude this subscription in final analysis.

The number of qualified terms in `subscription_terms_qualified` and `subscription_all_terms` should be the same.


In [0]:
%sql
select
count(distinct s.subscription_term_id) as n_terms_in_qualified,
count(distinct t.subscription_term_id) as n_terms_in_all_terms
from ${eda.qual} s
join ${eda.terms} t
on s.subscription_id = t.subscription_id and t.term_number = 1

In [0]:
%sql
select
subscription_id,
count(distinct subscription_term_id) as n_first_term
from ${eda.terms}
where term_number = 1
group by 1
having n_first_term > 1

This is because 10 subscriptions have more than one “first” term. These subscriptions should be excluded from future analysis.

After excluding the 10 subscriptions, the cohort size is as below:

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW subscription_terms_qualified_new AS
SELECT 
*
from ${eda.qual}
where subscription_id not in (
    select
        subscription_id
    from ${eda.terms}
    where term_number = 1
    group by 1
    having count(distinct subscription_term_id) > 1
);

SELECT
    COUNT(*)                        AS total_terms,
    COUNT(DISTINCT subscription_id) AS unique_subscriptions
FROM subscription_terms_qualified_new

## 2. Cancellation status as of June 30, 2026

In [0]:
%sql
SELECT
    t.cancel_status_before_cutoff,
    COUNT(*) AS n,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
FROM subscription_terms_qualified_new q
left join ${eda.terms} t
on t.subscription_term_id = q.subscription_term_id
GROUP BY 1
ORDER BY 2 DESC

## 3. Cross-tab: term status as of June 30, 2026 vs July 2, 2026

The cancellation label is assigned as of June 30, 2026. Therefore, we would not expect many cancelled subscriptions to still have an active status. However, as of July 2, 2026 (data pulling date), some subscriptions labeled as non-cancelled may now have a cancelled status.          

In [0]:
%sql
SELECT
    t.cancel_status_before_cutoff,
    t.term_status,
    COUNT(*) AS n
FROM subscription_terms_qualified_new q
left join ${eda.terms} t
    ON t.subscription_term_id = q.subscription_term_id
GROUP BY 1, 2
ORDER BY 1, 3 DESC

### 3.1 Why are some subscriptions labeled "cancelled" but currently "active"?

**Explanation :** Their current term hasn't ended yet.

In the subscription_terms table, a term can still have term_status = 'active' after a cancellation request has been made, up until the term actually ends.

In [0]:
%sql
SELECT
count(distinct t.subscription_id)  n_not_ended_yet
FROM subscription_terms_qualified_new q
left join ${eda.terms} t
    ON t.subscription_term_id = q.subscription_term_id
where t.cancel_status_before_cutoff = 'cancelled' and t.term_status = 'active' and t.term_ended_at is null

The other 10 is because of errors in term_status, but no need to drop them.

In [0]:
%sql
SELECT
t.subscription_id,
t.subscription_term_id,
t.term_number,
t.term_started_at,
t.term_ended_at,
t.cancel_requested_at
FROM subscription_terms_qualified_new q
left join ${eda.terms} t
    ON t.subscription_term_id = q.subscription_term_id
where t.cancel_status_before_cutoff = 'cancelled' and t.term_status = 'active' and t.term_ended_at is not null

## 4. Term start date distribution — confirm all started before 2026-06-01

Note: Unactivated subscriptions may still appear in the subscription_terms table, but their term_started_at value is null.

In [0]:
%sql
SELECT
    DATE_TRUNC('month', t.term_started_at)::date AS term_start_month,
    COUNT(*) AS n
FROM subscription_terms_qualified_new q
left join ${eda.terms} t
    ON t.subscription_term_id = q.subscription_term_id
GROUP BY 1
ORDER BY 1 desc

## 5. Cancellation timing — when did cancelled subscribers cancel?

In [0]:
%sql
SELECT
    DATE_TRUNC('week', cancel_requested_at)::date AS cancel_week,
    COUNT(*) AS n
FROM subscription_terms_qualified_new q
left join ${eda.terms} t
    ON t.subscription_term_id = q.subscription_term_id
WHERE is_cancelled_before_cutoff is TRUE
GROUP BY 1
ORDER BY 1

## 6. Check for duplicates in cohort (should be one row per subscription_term_id)

In [0]:
%sql
SELECT 
COUNT(*) AS total_rows, 
COUNT(DISTINCT subscription_term_id) AS unique_terms
FROM subscription_terms_qualified_new

## 7. Label completeness — every qualified term should have a label

###7.1 Overall cancellation

In [0]:
%sql
SELECT
    COUNT(distinct q.subscription_term_id)   AS cohort_size,
    COUNT(distinct case when t.is_cancelled_before_cutoff is not null then t.subscription_term_id else null end)   AS labeled,
    COUNT(distinct q.subscription_term_id) - COUNT(distinct case when t.is_cancelled_before_cutoff is not null then t.subscription_term_id else null end) AS missing_labels
FROM subscription_terms_qualified_new q
left join ${eda.terms} t
    ON t.subscription_term_id = q.subscription_term_id

### 7.2 30-day cancellation after subs start

In [0]:
%sql
SELECT
    COUNT(q.subscription_term_id)   AS cohort_size,
    COUNT(l.subscription_term_id)   AS labeled,
    COUNT(q.subscription_term_id) - COUNT(l.subscription_term_id) AS missing_labels
FROM subscription_terms_qualified_new q
LEFT JOIN general_scratch_catalog.general_scratch.ed_silver_subscription_term_start_labels l
    ON q.subscription_term_id = l.subscription_term_id

## 8. Latest term plan

In the `plan_terms` table, reactivated subscriptions have more than one terms, and each term has its own latest plan term, identified by `is_latest_plan_term` = TRUE.

In [0]:
%sql
-- #subs with more than one latest terms
select
subscription_id,
count(distinct subscription_term_id) as n_latest_terms
from ${eda.plan_terms}
where is_latest_plan_term = TRUE
group by 1
having n_latest_terms > 1

In [0]:
%sql
select
subscription_id,
subscription_term_id,
subscription_plan_term_id,
term_number,
is_latest_plan_term
from ${eda.plan_terms}
where subscription_id = 'd84aecb2-0aec-4e0f-b1a2-4481f640aa08'

## 9. Does fellout users have a plan term?

In [0]:
%sql
select
*
from general_scratch_catalog.general_scratch.ed_bronze_subscription_plan_terms
where subscription_id in (
    select
    subscription_id
    from general_scratch_catalog.general_scratch.ed_bronze_subscriptions
    where is_activated = FALSE
)

##10. Label window sanity checks

Verifies:
- `day_1_after_start` = `term_started_at + 1 day` for all rows
- `day_30_after_start` = `term_started_at + 30 days` for all rows
- No row has `cancel_requested_at` outside the label window but incorrectly classified
- No label window extends past the label cutoff date (2026-06-30)
- No `day_1_after_start` exceeds the term start cutoff (2026-06-01)

All count queries should return 0.

In [0]:
%sql
-- day_1_after_start must equal term_started_at + 1 day — should return 0
SELECT COUNT(*) AS bad_day_1
FROM ${eda.labels}
WHERE day_1_after_start != DATEADD(DAY, 1, term_started_at)

In [0]:
%sql
-- day_30_after_start must equal term_started_at + 30 days — should return 0
SELECT COUNT(*) AS bad_day_30
FROM ${eda.labels}
WHERE day_30_after_start != DATEADD(DAY, 30, term_started_at)

In [0]:
%sql
-- cancelled_in_30_days: cancel_requested_at must be within [day_1, day_30] — should return 0
SELECT COUNT(*) AS misclassified_cancelled
FROM ${eda.labels}
WHERE cancel_status = 'cancelled_in_30_days'
  AND (cancel_requested_at < day_1_after_start OR cancel_requested_at > day_30_after_start)

In [0]:
%sql
-- cancelled_at_start: cancel_requested_at must equal term_started_at — should return 0
SELECT COUNT(*) AS misclassified_at_start
FROM ${eda.labels}
WHERE cancel_status = 'cancelled_at_start'
  AND cancel_requested_at != term_started_at

In [0]:
%sql
-- No day_30_after_start should exceed the data pull date (2026-06-30) — should return 0
SELECT COUNT(*) AS window_exceeds_pull_date
FROM ${eda.labels}
WHERE day_30_after_start > '2026-06-30'

In [0]:
%sql
-- No day_1_after_start should exceed the data cutoff (2026-06-01) — should return 0
SELECT COUNT(*) AS day_1_exceeds_cutoff
FROM ${eda.labels}
WHERE day_1_after_start > '2026-06-01'

In [0]:
%sql
-- Spot check: sample rows from cancelled cases with their window dates
SELECT
    cancel_status,
    term_started_at,
    day_1_after_start,
    day_30_after_start,
    cancel_requested_at
FROM ${eda.labels}
WHERE cancel_status != 'not_cancelled'
LIMIT 10

## Weekly snapshot table sanity checks — ed_silver_subscription_weekly_snapshots

All count queries below should return 0.

- One row per (subscription_id, snapshot_date) — no duplicates
- Snapshots are weekly: `snapshot_date = term_started_at + k * 7`
- Churners: no snapshot on or after `cancel_requested_at`
- Non-churners: label window end (`snapshot + 30`) ≤ 2026-06-30
- Label = 1 only when `cancel_requested_at` falls within `[snapshot + 1, snapshot + 30]`

In [ ]:
SNAPSHOTS = "general_scratch_catalog.general_scratch.ed_silver_subscription_weekly_snapshots"
spark.conf.set("eda.snapshots", SNAPSHOTS)

In [ ]:
%sql
-- Row count and unique (subscription, snapshot) pairs — should be equal (no duplicates)
SELECT
    COUNT(*)                                                AS total_rows,
    COUNT(DISTINCT subscription_id || '|' || snapshot_date) AS unique_sub_snapshot_pairs,
    COUNT(*) - COUNT(DISTINCT subscription_id || '|' || snapshot_date) AS duplicates
FROM ${eda.snapshots}

In [ ]:
%sql
-- Snapshots are weekly: (snapshot_date - term_started_at) must be divisible by 7 — should return 0
SELECT COUNT(*) AS non_weekly_snapshots
FROM ${eda.snapshots}
WHERE DATEDIFF(DAY, term_started_at, snapshot_date) % 7 != 0

In [ ]:
%sql
-- Churners: no snapshot should be on or after cancel_requested_at — should return 0
SELECT COUNT(*) AS snapshot_on_or_after_cancel
FROM ${eda.snapshots}
WHERE cancel_requested_at IS NOT NULL
  AND snapshot_date >= cancel_requested_at

In [ ]:
%sql
-- Non-churners: label window end (snapshot + 30) must not exceed 2026-06-30 — should return 0
SELECT COUNT(*) AS window_exceeds_label_cutoff
FROM ${eda.snapshots}
WHERE cancel_requested_at IS NULL
  AND DATEADD(DAY, 30, snapshot_date) > '2026-06-30'

In [ ]:
%sql
-- Label = 1 must only occur when cancel is within [snapshot+1, snapshot+30] — should return 0
SELECT COUNT(*) AS misclassified_label_1
FROM ${eda.snapshots}
WHERE label = 1
  AND (cancel_requested_at IS NULL
       OR cancel_requested_at <= snapshot_date
       OR cancel_requested_at > DATEADD(DAY, 30, snapshot_date))

In [ ]:
%sql
-- Label = 0 must not occur when cancel falls within [snapshot+1, snapshot+30] — should return 0
SELECT COUNT(*) AS misclassified_label_0
FROM ${eda.snapshots}
WHERE label = 0
  AND cancel_requested_at > snapshot_date
  AND cancel_requested_at <= DATEADD(DAY, 30, snapshot_date)

In [ ]:
%sql
-- Overall label distribution and label rate by week (spot check)
SELECT
    week_number,
    COUNT(*) AS n_snapshots,
    SUM(label) AS n_label_1,
    ROUND(SUM(label) * 100.0 / COUNT(*), 1) AS label_rate_pct
FROM ${eda.snapshots}
GROUP BY 1
ORDER BY 1

### Additional check: involuntary churners excluded from weekly snapshots

Subscribers with `cancel_requested_at IS NULL` AND `term_ended_at <= '2026-06-30'` are involuntary churners and should NOT appear in the snapshot table.

All count queries below should return 0.

In [ ]:
%sql
-- No snapshot should belong to an involuntary churner
-- (cancel_requested_at IS NULL AND term_ended_at <= 2026-06-30) — should return 0
SELECT COUNT(*) AS involuntary_churners_in_snapshots
FROM ${eda.snapshots} s
JOIN general_scratch_catalog.general_scratch.ed_bronze_subscription_terms t
    ON s.subscription_term_id = t.subscription_term_id
WHERE t.cancel_requested_at IS NULL
  AND t.term_ended_at IS NOT NULL
  AND t.term_ended_at::date <= '2026-06-30'

In [ ]:
%sql
-- Confirm: all label=0 subscribers in snapshots are either truly retained
-- (term_ended_at IS NULL or > 2026-06-30) OR voluntary churners (cancel_requested_at IS NOT NULL)
-- Should return 0 — no label=0 rows should be involuntary churners
SELECT COUNT(*) AS label_0_involuntary_churners
FROM ${eda.snapshots} s
JOIN general_scratch_catalog.general_scratch.ed_bronze_subscription_terms t
    ON s.subscription_term_id = t.subscription_term_id
WHERE s.label = 0
  AND t.cancel_requested_at IS NULL
  AND t.term_ended_at IS NOT NULL
  AND t.term_ended_at::date <= '2026-06-30'

In [ ]:
%sql
-- Breakdown: retained vs voluntary churner composition of the snapshot table (spot check)
SELECT
    CASE
        WHEN s.cancel_requested_at IS NOT NULL THEN 'voluntary_churner'
        ELSE 'retained'
    END AS subscriber_type,
    COUNT(DISTINCT s.subscription_id) AS n_subscribers,
    COUNT(*) AS n_snapshots,
    SUM(s.label) AS n_label_1
FROM ${eda.snapshots} s
GROUP BY 1
ORDER BY 2 DESC